# Simulation

Discrete-event simulation follows individual customers through a queue. Instead of advancing a clock in fixed steps, it jumps between arrivals, service completions and other state-changing events. OpenQTSim uses SimPy to schedule these events and manage resources.

Read [Queueing theory](0001%20-%20Queueing%20theory.ipynb) first for notation. This notebook introduces the simulation logic; the next chapter implements queueing examples using OpenQTSim.

## From durations to events

For a single first-come-first-served server with no abandonment, define arrival time $AT_i$, service duration $ST_i$, service start $TSB_i$ and service end $TSE_i$. Interarrival durations determine $AT_i$ by cumulative summation. Starting empty:

$$TSB_i=\max(AT_i,TSE_{i-1}),\quad TSE_i=TSB_i+ST_i,\quad WQ_i=TSB_i-AT_i.$$

The small calculation below illustrates this event logic without introducing random sampling. It is a teaching calculation, not a replacement for OpenQTSim's multi-resource event engine.

In [1]:
import pandas as pd
from IPython.display import display

arrivals = [0.0, 1.0, 4.0]
service_times = [2.0, 2.0, 1.0]
previous_end = 0.0
rows = []
for customer, (arrival, service) in enumerate(zip(arrivals, service_times), start=1):
    start = max(arrival, previous_end)
    end = start + service
    rows.append({'customer': customer, 'arrival': arrival, 'service start': start,
                 'service end': end, 'waiting': start - arrival})
    previous_end = end
event_table = pd.DataFrame(rows).set_index('customer')
display(event_table)
print(f'Mean waiting time: {event_table["waiting"].mean():.3f} hours')


,arrival,service start,service end,waiting
customer,,,,
1,0.0,0.0,2.0,0.0
2,1.0,2.0,4.0,1.0
3,4.0,4.0,5.0,0.0


Mean waiting time: 0.333 hours


## Building an OpenQTSim experiment

1. Specify demand, arrival and service distributions, units and resource constraints.
2. Construct customer and station classes from OpenQTSim mixins; `create_object` builds the class and `inspect_object` describes its inputs.
3. Assemble a `Queue` and pass it to a `Simulation`, with separate arrival and service seeds.
4. Run the required number of arrivals and inspect the customer log and summary statistics.
5. Compare against a deterministic hand calculation or analytical reference before interpreting a more complex model.

The [D/D/1 basic example](1001%20-%20D%20D%201%20-%20basics%20of%20queueing.ipynb) walks through these steps. Shared-quay examples then add concurrent resource and length constraints. Those constraints cannot generally be represented by the single-server recurrence above.

## Randomness, replications and uncertainty

A seed makes a random stream reproducible; it does not make one realisation representative of steady state. Use separate streams for arrivals and services. The parallel task helpers split a master seed into task-specific arrival and service streams.

**Run length** is the number of arrivals in one realisation. **Replications** are independent realisations of the same experiment. A longer run can reduce start-up influence and sampling noise; more independent replications let us estimate between-run uncertainty. Customers within one run are correlated, so treating all waiting times as independent observations can understate uncertainty.

Runs start empty and finish serving the accepted arrivals. Compare increasing run lengths and inspect stability before claiming steady-state performance. Replication confidence intervals describe sampling error under the model, not model validity or uncertainty in future demand. Dask distributes independent experiments; adding workers changes computation time, not the underlying statistical assumptions.

## Interpreting the output

Inspect the event log as well as aggregate means: verify service ordering, capacity, conservation of customers and time units. Distinguish averages over arriving customers from time averages over the simulated horizon, particularly when a run begins empty and ends with a drain period. Report any reneging separately rather than counting abandoned customers as successfully served.

The book displays saved outputs and does not execute simulations while building. To refresh a result, run and save the original notebook in a suitable Python environment, then rebuild the book.